<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/bioassay/bioassay_lean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bioassay: Bayesian workflow

**Short Bayesian course — worked example**

$$
\text{data}
\rightarrow
\text{model}
\rightarrow
\text{prior predictive}
\rightarrow
\text{fit}
\rightarrow
LD50
\rightarrow
\text{posterior predictive}
$$

## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import pymc as pm
import arviz_plots as azp
import arviz_stats as azs

RANDOM_SEED = 20260923
azp.style.use("arviz-variat")

# Figures used in the slides: Slides/figures/<notebook>_<section>[_<n>].svg
NOTEBOOK = "bioassay_lean"
FIG_DIR = Path("../Slides/figures") if Path("../Slides").is_dir() else Path("figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

def save_slide_figure(fig, section, number=None):
    """Save a Matplotlib figure or ArviZ PlotCollection for the slides."""
    suffix = f"_{number}" if number else ""
    fig.savefig(
        FIG_DIR / f"{NOTEBOOK}_{section}{suffix}.svg", bbox_inches="tight", transparent=True
    )

def add_predictive_legend(ax):
    """Label the two predictive HDIs, predictive median, and observations."""
    ax.legend(
        handles=[
            Line2D([0], [0], color="C1", lw=2, label="Predictive median"),
            Patch(facecolor="C0", alpha=1.0, label="50% HDI"),
            Patch(facecolor="C0", alpha=0.32, label="90% HDI"),
            Line2D(
                [0], [0], marker="o", linestyle="none", color="black", label="Observed deaths"
            ),
        ],
        loc="lower center",
        bbox_to_anchor=(0.5, 1.02),
        ncols=2,
        frameon=False,
    )

print("PyMC:", pm.__version__)
print("arviz-plots:", azp.__version__)
print("arviz-stats:", azs.__version__)

## 1. Data

In [ ]:
dose = np.array([-0.86, -0.30, -0.05, 0.73])
n = np.array([5, 5, 5, 5])
deaths = np.array([0, 1, 3, 5])

bioassay = pd.DataFrame(
    {
        "dose_log_g_ml": dose,
        "animals": n,
        "deaths": deaths,
    }
)
bioassay["proportion_dead"] = bioassay["deaths"] / bioassay["animals"]
bioassay

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(dose, deaths / n, s=70)
ax.set(
    xlabel="Dose log(g/ml)",
    ylabel="Observed proportion dead",
    ylim=(-0.05, 1.05),
)
plt.show()

## 2. Model

$$
y_j\sim\operatorname{Binomial}(n_j,p_j),
\qquad
\operatorname{logit}(p_j)=\alpha+\beta x_j
$$

$$
\alpha\sim N(0,5),
\qquad
\beta\sim\operatorname{HalfNormal}(5).
$$

In [ ]:
coords = {"dose_log_g_ml": dose}

with pm.Model(coords=coords) as model:
    dose_data = pm.Data("dose", dose, dims="dose_log_g_ml")

    alpha = pm.Normal("alpha", mu=0, sigma=5)
    beta = pm.HalfNormal("beta", sigma=5)

    logit_p = alpha + beta * dose_data

    ld50_log_g_ml = pm.Deterministic(
        "LD50_log_g_ml",
        -alpha / beta,
    )
    ld50_mg_ml = pm.Deterministic(
        "LD50_mg_ml",
        1000 * pm.math.exp(ld50_log_g_ml),
    )

    pm.Binomial(
        "deaths",
        n=n,
        logit_p=logit_p,
        observed=deaths,
        dims="dose_log_g_ml",
    )

## 3. Prior predictive

**Question:** Before fitting, what death counts do these priors say are plausible?

The bands below summarize replicated **death counts** $y$, not the latent mortality probability $p$.

In [ ]:
with model:
    prior = pm.sample_prior_predictive(
        draws=1000,
        var_names=["deaths"],
        random_seed=RANDOM_SEED,
    )

In [ ]:
azp.plot_lm(
    prior,
    x="dose",
    y="deaths",
    y_obs="deaths",
    group="prior_predictive",
    plot_dim="dose_log_g_ml",
    ci_prob=(0.50, 0.90),
    ci_kind="hdi",
    point_estimate="median",
    smooth=False,
    visuals={
        "pe_line": {"color": "C1"},
        "ci_band": {"color": "C0"},
        "observed_scatter": {"color": "black", "alpha": 1},
    },
)

ax = plt.gca()
ax.set(xlabel="Dose log(g/ml)", ylabel="Deaths out of 5", ylim=(-0.25, 5.25))
add_predictive_legend(ax)
save_slide_figure(plt.gcf(), "prior-predictive")
plt.show()

## 4. Fit and diagnose

In [ ]:
with model:
    idata = pm.sample(
        draws=1000,
        tune=1500,
        chains=4,
        nuts={"target_accept": 0.90},
        random_seed=RANDOM_SEED,
    )

In [ ]:
print("Divergences:", idata["sample_stats"]["diverging"].sum().item())

azs.summary(
    idata,
    var_names=["alpha", "beta"],
    ci_prob=0.90,
    ci_kind="hdi",
    round_to=2,
)

In [ ]:
azp.plot_trace_dist(
    idata,
    var_names=["alpha", "beta"],
)

## 5. LD50

$$
LD50=-\frac{\alpha}{\beta}.
$$

It is stored as a PyMC deterministic quantity.

In [ ]:
azs.summary(
    idata,
    var_names=["LD50_log_g_ml", "LD50_mg_ml"],
    ci_prob=0.90,
    ci_kind="hdi",
    round_to=2,
)

In [ ]:
azp.plot_dist(
    idata,
    var_names=["LD50_mg_ml"],
    point_estimate="median",
    ci_prob=0.90,
    ci_kind="hdi",
)

## 6. Posterior predictive

**Question:** After fitting, can the model generate death counts like those observed?

This has the same format as the prior predictive plot. The difference is conditioning: these replicated counts are generated from the posterior rather than the prior.

In [ ]:
with model:
    pm.sample_posterior_predictive(
        idata,
        var_names=["deaths"],
        extend_inferencedata=True,
        random_seed=RANDOM_SEED,
    )

In [ ]:
azp.plot_lm(
    idata,
    x="dose",
    y="deaths",
    y_obs="deaths",
    group="posterior_predictive",
    plot_dim="dose_log_g_ml",
    ci_prob=(0.50, 0.90),
    ci_kind="hdi",
    point_estimate="median",
    smooth=False,
    visuals={
        "pe_line": {"color": "C1"},
        "ci_band": {"color": "C0"},
        "observed_scatter": {"color": "black", "alpha": 1},
    },
)

ax = plt.gca()
ax.set(xlabel="Dose log(g/ml)", ylabel="Deaths out of 5", ylim=(-0.25, 5.25))
add_predictive_legend(ax)
save_slide_figure(plt.gcf(), "posterior-predictive")
plt.show()

## 7. Explore

- Remove the positive-slope constraint.
- Change the priors and rerun the prior predictive check.

Source: Gelman & Vehtari, *Bayesian Workflow*, §3.5.